# 06: Figures (Phase 6) and dashboard exports (Phase 7)

**Phase 6 goal:** the 13 numbered report figures, saved as 300 dpi PNGs in `figures/`, with a caption and a one-sentence takeaway each (`figures/captions.md`). Every takeaway is computed from the data in this notebook.

**Style rules** (`src/plots.py`): one colour and marker per case company in every chart (Nvidia green circle, AMD red square, Intel blue triangle, Wolfspeed amber diamond; Wolfspeed's colour changed from the guide's grey because grey failed the colour-blind check), direct end labels, one y-axis per panel, label year on the x-axis, units on every axis.

**Phase 7 goal** (last section): the CSV exports for the Power BI dashboard in `dashboard/exports/`, checked against the earlier tables; the build steps are in `dashboard/BUILD_GUIDE.md`.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
from sklearn.metrics import roc_curve, roc_auc_score

sys.path.insert(0, "..")
from src import config
from src import plots as P
from src.models import FEATURES, TARGET

P.apply_style()
FIG = config.FIGURES_DIR
pd.set_option("display.width", 200)

ratios = pd.read_csv(config.PROCESSED_DIR / "ratios.csv")
zs = pd.read_csv(config.PROCESSED_DIR / "zscores.csv")
comparison = pd.read_csv(config.PROCESSED_DIR / "model_comparison.csv")
test_pred = pd.read_csv(config.PROCESSED_DIR / "test_predictions.csv")
importance = pd.read_csv(config.PROCESSED_DIR / "feature_importance.csv")
wf = pd.read_csv(config.PROCESSED_DIR / "walk_forward_predictions.csv")

analysis = ratios[ratios["analysis_year"]]
case = analysis[analysis["cik"].isin(config.CASE_COMPANIES)].copy()
case["company"] = case["cik"].map(config.CASE_COMPANIES)
case = case.sort_values(["company", "label_year"])
YEARS = list(range(config.LABEL_YEAR_START, config.LABEL_YEAR_END + 1))
captions = []   # (number, file, title, takeaway)

def series(df, col, company):
    d = df[df["company"] == company].set_index("label_year")[col].reindex(YEARS)
    return d

def lines(ax, df, col, scale=1.0):
    ends = {}
    for c in P.COMPANY_ORDER:
        s = series(df, col, c) * scale
        P.company_line(ax, s.index, s.values, c)
        last = s.dropna()
        if len(last):
            ends[c] = (last.index[-1], last.iloc[-1])
    return ends

pct = FuncFormatter(lambda v, _: f"{v:.0f}%")
PERSIST = "Baseline: persistence (Z' < 1.23 now)"  # model name used in the results files
print("Ready:", len(ratios), "ratio rows;", len(case), "case-company rows")

Ready: 1028 ratio rows; 44 case-company rows


## Figure 1: Companies per year

In [2]:
counts = analysis.groupby(["label_year", "low_revenue"]).size().unstack(fill_value=0).reindex(YEARS)
fig, ax = plt.subplots(figsize=(7.2, 3.6))
op, lowrev = counts[False], counts[True]
ax.bar(YEARS, op, width=0.62, color="#2a78d6", label="Operating companies (revenue ≥ USD 10m)", edgecolor=P.SURFACE, lw=1)
ax.bar(YEARS, lowrev, width=0.62, bottom=op, color="#b9b7ae", label="Revenue below USD 10m", edgecolor=P.SURFACE, lw=1)
for y, t in zip(YEARS, op + lowrev):
    ax.annotate(f"{t}", (y, t), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8, color=P.INK_2)
ax.set_title("SIC 3674 companies with a clean annual filing, by label year")
ax.set_ylabel("Companies (count)")
ax.set_xticks(YEARS)
ax.set_xlabel("Label year")
ax.grid(axis="x", visible=False)
ax.set_ylim(0, (op + lowrev).max() * 1.15)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
P.save(fig, FIG / "fig01_companies_per_year.png")
tot = op + lowrev
captions.append((1, "fig01_companies_per_year.png", "Number of SIC 3674 companies per label year",
    f"The sample shrinks from {tot.iloc[0]} companies in {YEARS[0]} to {tot.iloc[-1]} in {YEARS[-1]}; "
    f"{int(lowrev.sum())} of {int(tot.sum())} company-years are very small companies (revenue below USD 10 million), which are excluded from modelling."))
counts

low_revenue,False,True
label_year,,
2015,82,11
2016,77,6
2017,77,9
2018,75,10
2019,72,11
2020,71,10
2021,76,13
2022,73,10
2023,73,8


## Figure 2: Revenue

In [3]:
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ends = lines(ax, case, "revenue")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_title("Revenue of the four case companies")
ax.set_ylabel("Revenue (USD millions, log scale)")
P.year_axis(ax, YEARS)
P.end_labels(ax, ends)
P.save(fig, FIG / "fig02_revenue.png")
r = case.pivot_table(index="label_year", columns="company", values="revenue")
captions.append((2, "fig02_revenue.png", "Revenue, 2015 to 2025 (log scale)",
    f"Nvidia's revenue grew from USD {r.loc[2015,'Nvidia']:,.0f}m to {r.loc[2025,'Nvidia']:,.0f}m and overtook Intel's in "
    f"{int(r.index[(r['Nvidia'] > r['Intel'])][0])}; Intel fell from a peak of USD {r['Intel'].max():,.0f}m ({int(r['Intel'].idxmax())}) to {r.loc[2025,'Intel']:,.0f}m, "
    f"and Wolfspeed shrank from USD {r.loc[2015,'Wolfspeed']:,.0f}m to {r.loc[2025,'Wolfspeed']:,.0f}m. A log scale is used because Nvidia's 2025 revenue is about "
    f"{r.loc[2025,'Nvidia'] / r.loc[2025,'Wolfspeed']:,.0f} times Wolfspeed's."))

## Figure 3: Net income (profit and loss years)

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.2))
for ax, c in zip(axes.ravel(), P.COMPANY_ORDER):
    s = series(case, "net_income", c)
    ax.bar(s.index, s.values, width=0.62, color=P.COMPANY_STYLE[c]["color"], edgecolor=P.SURFACE, lw=1)
    ax.axhline(0, color=P.INK_2, lw=0.8)
    losses = int((s < 0).sum())
    ax.set_title(f"{c}: {losses} loss year{'s' if losses != 1 else ''}")
    ax.set_xticks(YEARS[::2])
    ax.grid(axis="x", visible=False)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
for ax in axes[:, 0]:
    ax.set_ylabel("Net income (USD millions)")
for ax in axes[1, :]:
    ax.set_xlabel("Label year")
fig.suptitle("Net income of the four case companies (bars below zero are losses; each panel has its own scale)",
             x=0.01, ha="left", fontsize=10, fontweight="bold")
fig.tight_layout()
P.save(fig, FIG / "fig03_net_income.png")
ni = case.pivot_table(index="label_year", columns="company", values="net_income")
loss_years = {c: [int(y) for y in ni.index[ni[c] < 0]] for c in P.COMPANY_ORDER}
captions.append((3, "fig03_net_income.png", "Net income, 2015 to 2025 (separate scales)",
    f"Nvidia was profitable in every year; AMD made losses in {', '.join(map(str, loss_years['AMD']))} and profits afterwards; Intel turned to losses in "
    f"{' and '.join(map(str, loss_years['Intel']))} (USD {ni.loc[2024,'Intel']:,.0f}m in 2024); Wolfspeed lost money in every year from 2015 to 2025, "
    f"reaching USD {ni.loc[2025,'Wolfspeed']:,.0f}m in 2025."))
loss_years

{'Nvidia': [],
 'AMD': [2015, 2016],
 'Intel': [2024, 2025],
 'Wolfspeed': [2015,
  2016,
  2017,
  2018,
  2019,
  2020,
  2021,
  2022,
  2023,
  2024,
  2025]}

## Figure 4: Liquidity (current and quick ratio)

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.6), sharey=False)
for ax, col, title in zip(axes, ["current_ratio", "quick_ratio"], ["Current ratio (dashed line = 1)", "Quick ratio (dashed line = 1)"]):
    ends = lines(ax, case, col)
    P.ref_line(ax, 1, None)
    ax.set_title(title)
    ax.set_ylabel("Ratio (times)")
    P.year_axis(ax, YEARS)
    ax.set_ylim(bottom=0)
    P.end_labels(ax, ends)
fig.tight_layout()
P.save(fig, FIG / "fig04_liquidity.png")
cr = case.pivot_table(index="label_year", columns="company", values="current_ratio")
captions.append((4, "fig04_liquidity.png", "Current ratio and quick ratio, with a reference line at 1",
    f"All four companies kept a current ratio above 1 in every year until Wolfspeed fell to {cr.loc[2025,'Wolfspeed']:.2f} in 2025, when its current "
    f"liabilities rose from USD 665m to 7,090m; Wolfspeed's current ratio was still {cr.loc[2024,'Wolfspeed']:.2f} in 2024, so liquidity gave no early warning of its distress."))

## Figure 5: Leverage (debt-to-equity and interest coverage)

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.8))
ax = axes[0]
ends = lines(ax, case, "debt_to_equity")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_title("Debt-to-equity")
ax.set_ylabel("Total liabilities / equity (times, log scale)")
P.year_axis(ax, YEARS)
gaps = [f"{c} {y}" for c, y in [("AMD", 2015), ("Wolfspeed", 2025)] if pd.isna(series(case, "debt_to_equity", c).get(y))]
ax.annotate("No point where equity was negative: " + ", ".join(gaps), xy=(0, -0.27), xycoords="axes fraction",
            fontsize=7.5, color=P.INK_2)
P.end_labels(ax, ends)

ax = axes[1]
ends = lines(ax, case, "interest_coverage")
ax.set_yscale("symlog", linthresh=1)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
P.ref_line(ax, 1.5, None)
ax.axhline(0, color=P.AXIS, lw=0.8)
ax.set_title("Interest coverage (dashed line = 1.5)")
ax.set_ylabel("Operating income / interest (times, symlog)")
P.year_axis(ax, YEARS)
P.end_labels(ax, ends)
fig.tight_layout()
P.save(fig, FIG / "fig05_leverage.png")
de = case.pivot_table(index="label_year", columns="company", values="debt_to_equity")
ic = case.pivot_table(index="label_year", columns="company", values="interest_coverage")
captions.append((5, "fig05_leverage.png", "Debt-to-equity (log scale) and interest coverage (symmetric log scale)",
    f"Intel's interest coverage fell from {ic.loc[2021,'Intel']:.1f} (2021) to {ic.loc[2022,'Intel']:.1f} (2022) and {ic.loc[2023,'Intel']:.2f} (2023) before its 2024 loss, "
    f"and Wolfspeed's debt-to-equity rose from {de.loc[2022,'Wolfspeed']:.2f} (2022) to {de.loc[2024,'Wolfspeed']:.2f} (2024); AMD's leverage fell from "
    f"{de.loc[2016,'AMD']:.1f} in 2016 to {de.loc[2025,'AMD']:.2f} in 2025. Blank points mark negative equity (AMD 2015, Wolfspeed 2025). "
    f"Wolfspeed's coverage for 2015 to 2019 uses cash interest paid as a substitute."))

## Figure 6: Margins

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(7.4, 3.6))
for ax, col, title in zip(axes, ["gross_margin", "operating_margin", "net_margin"], ["Gross margin", "Operating margin", "Net margin"]):
    ends = lines(ax, case, col, scale=100)
    ax.axhline(0, color=P.AXIS, lw=0.8)
    ax.set_title(title)
    ax.yaxis.set_major_formatter(pct)
    P.year_axis(ax, YEARS)
    ax.set_xticks(YEARS[::5]); ax.set_xticklabels([str(y) for y in YEARS[::5]])
    P.end_labels(ax, ends)
axes[0].set_ylabel("Share of revenue (%)")
fig.tight_layout()
P.save(fig, FIG / "fig06_margins.png")
gm = case.pivot_table(index="label_year", columns="company", values="gross_margin") * 100
om = case.pivot_table(index="label_year", columns="company", values="operating_margin") * 100
captions.append((6, "fig06_margins.png", "Gross, operating and net margin (percent of revenue)",
    f"Nvidia's operating margin rose from {om.loc[2015,'Nvidia']:.0f}% to {om.loc[2024,'Nvidia']:.0f}% (2024); Intel's gross margin fell from "
    f"{gm.loc[2015,'Intel']:.0f}% to {gm.loc[2024,'Intel']:.0f}% (2024); Wolfspeed's gross margin turned negative in 2025 ({gm.loc[2025,'Wolfspeed']:.0f}%), "
    f"and its operating margin was negative in every year from 2015."))

## Figure 7: Returns (ROA and ROE)

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.6))
for ax, col, title in zip(axes, ["roa", "roe"], ["Return on assets (ROA)", "Return on equity (ROE)"]):
    ends = lines(ax, case, col, scale=100)
    ax.axhline(0, color=P.AXIS, lw=0.8)
    ax.set_title(title)
    ax.yaxis.set_major_formatter(pct)
    ax.set_ylabel("Net income / " + ("total assets" if col == "roa" else "equity") + " (%)")
    P.year_axis(ax, YEARS)
    P.end_labels(ax, ends)
fig.tight_layout()
P.save(fig, FIG / "fig07_returns.png")
roe = case.pivot_table(index="label_year", columns="company", values="roe") * 100
roa = case.pivot_table(index="label_year", columns="company", values="roa") * 100
captions.append((7, "fig07_returns.png", "Return on assets and return on equity (percent)",
    f"Nvidia's ROE reached {roe['Nvidia'].max():.0f}% ({int(roe['Nvidia'].idxmax())}) and its ROA {roa['Nvidia'].max():.0f}%; AMD's ROA dropped from "
    f"{roa.loc[2021,'AMD']:.0f}% (2021) to {roa.loc[2022,'AMD']:.0f}% (2022) as its total assets rose from USD "
    f"{case.set_index(['company','label_year']).loc[('AMD',2021),'total_assets']:,.0f}m to "
    f"{case.set_index(['company','label_year']).loc[('AMD',2022),'total_assets']:,.0f}m with the Xilinx acquisition; ROE is blank where equity "
    f"was negative (AMD 2015, Wolfspeed 2025)."))

## Figure 8: Altman Z' score

In [9]:
z = zs[zs["cik"].isin(config.CASE_COMPANIES) & zs["label_year"].between(YEARS[0], YEARS[-1])].copy()
z["company"] = z["cik"].map(config.CASE_COMPANIES)
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ends = lines(ax, z, "z_prime")
lo, hi = min(-2, np.nanmin(z["z_prime"]) - 0.3), max(6, np.nanmax(z["z_prime"]) + 0.3)
ax.axhspan(lo, 1.23, color="#ebe9e2", zorder=0, lw=0)
ax.axhspan(1.23, 2.90, color="#f4f3ef", zorder=0, lw=0)
P.ref_line(ax, 1.23, "1.23", va="top")
P.ref_line(ax, 2.90, "2.90")
for yv, txt in [((lo + 1.23) / 2, "Distress zone"), ((1.23 + 2.90) / 2, "Grey zone"), ((2.90 + hi) / 2, "Safe zone")]:
    ax.annotate(txt, xy=(0.995, yv), xycoords=("axes fraction", "data"), ha="right", va="center", fontsize=8,
                color=P.MUTED, fontstyle="italic")
ax.set_ylim(lo, hi)
ax.set_title("Altman Z' score of the four case companies")
ax.set_ylabel("Z' score")
P.year_axis(ax, YEARS)
ax.set_xlim(YEARS[0] - 0.4, YEARS[-1] + 2.4)
P.end_labels(ax, ends)
P.save(fig, FIG / "fig08_zscore.png")
zp = z.pivot_table(index="label_year", columns="company", values="z_prime")
first_distress = {c: next((int(y) for y in zp.index if zp.loc[y, c] < 1.23), None) for c in P.COMPANY_ORDER}
captions.append((8, "fig08_zscore.png", "Altman Z' score with the distress (1.23) and safe (2.90) cutoffs",
    f"Wolfspeed entered the distress zone in {first_distress['Wolfspeed']} and Intel in {first_distress['Intel']}; AMD left it after 2019 "
    f"(Z' {zp.loc[2019,'AMD']:.2f} to {zp.loc[2020,'AMD']:.2f}); Nvidia never fell below 1.23 but was in the grey zone in "
    f"{', '.join(str(int(y)) for y in zp.index if 1.23 <= zp.loc[y,'Nvidia'] < 2.90)}, contrary to the expectation that it would stay far above 2.90."))
first_distress

{'Nvidia': None, 'AMD': 2015, 'Intel': 2023, 'Wolfspeed': 2020}

## Figure 9: Correlation between ratios

In [10]:
ratio_cols = ["current_ratio", "quick_ratio", "cash_ratio", "debt_to_equity", "debt_ratio", "interest_coverage", "gross_margin",
              "operating_margin", "net_margin", "roa", "roe", "asset_turnover", "revenue_growth", "net_income_growth", "log_total_assets"]
nice = {"current_ratio": "Current ratio", "quick_ratio": "Quick ratio", "cash_ratio": "Cash ratio", "debt_to_equity": "Debt-to-equity",
        "debt_ratio": "Debt ratio", "interest_coverage": "Interest coverage", "gross_margin": "Gross margin", "operating_margin": "Operating margin",
        "net_margin": "Net margin", "roa": "ROA", "roe": "ROE", "asset_turnover": "Asset turnover", "revenue_growth": "Revenue growth",
        "net_income_growth": "Net income growth", "log_total_assets": "Log total assets"}
ops = analysis[~analysis["low_revenue"]]
corr = ops[ratio_cols].corr(method="spearman")
sub = corr.iloc[1:, :-1]                      # lower triangle without the diagonal of 1s
mask = np.triu(np.ones_like(sub, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(7.2, 6.4))
im = ax.imshow(np.ma.masked_array(sub.values, mask), cmap=P.DIVERGING, vmin=-1, vmax=1)
ax.set_xticks(range(sub.shape[1])); ax.set_xticklabels([nice[c] for c in sub.columns], rotation=55, ha="right")
ax.set_yticks(range(sub.shape[0])); ax.set_yticklabels([nice[c] for c in sub.index])
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
for i in range(sub.shape[0]):
    for j in range(i + 1):
        v = sub.values[i, j]
        if abs(v) >= 0.6:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="#ffffff" if abs(v) > 0.8 else P.INK)
cb = fig.colorbar(im, ax=ax, shrink=0.7)
cb.set_label("Spearman correlation"); cb.outline.set_visible(False)
ax.set_title(f"Correlation between ratios (operating companies, {YEARS[0]}-{YEARS[-1]}; values shown where |r| ≥ 0.6)")
P.save(fig, FIG / "fig09_correlation.png")
pairs = corr.where(~np.triu(np.ones_like(corr, dtype=bool))).stack().sort_values(key=abs, ascending=False)
top = pairs.head(5)
captions.append((9, "fig09_correlation.png", "Spearman correlation between the 15 ratios (operating companies, 2015 to 2025)",
    "The strongest correlations are " + "; ".join(f"{nice[a]} and {nice[b]} ({v:.2f})" for (a, b), v in top.items()) +
    ". These groups carry overlapping information, which is why logistic regression coefficients for related ratios should not be read one by one; "
    f"{len(ops):,} company-years are used."))
top

debt_ratio   debt_to_equity    0.994548
roe          roa               0.978689
roa          net_margin        0.974620
quick_ratio  current_ratio     0.972468
roe          net_margin        0.951604
dtype: float64

## Figure 10: Model comparison

In [11]:
t = comparison[comparison["set"] == "test"].copy()
short = {"Baseline: current ratio < 1": "Baseline: current ratio < 1", "Baseline: persistence (Z' < 1.23 now)": "Baseline: persistence"}
t["label"] = t["model"].map(lambda m: short.get(m, m))
t = t.sort_values("f1")
fig, axes = plt.subplots(1, 3, figsize=(7.4, 3.2), sharey=True)
for ax, metric, title in zip(axes, ["f1", "roc_auc", "recall"], ["F1 (distressed)", "ROC-AUC", "Recall (distressed)"]):
    colors = ["#2a78d6" if m == "Random forest" else (P.MUTED if m.startswith("Baseline") else "#9fc3ee") for m in t["model"]]
    ax.barh(t["label"], t[metric], height=0.6, color=colors)
    for yv, v in enumerate(t[metric]):
        ax.annotate(f"{v:.2f}", (v, yv), xytext=(3, 0), textcoords="offset points", va="center", fontsize=7.5, color=P.INK_2)
    ax.set_xlim(0, 1.15)
    ax.set_title(title)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Score (0-1)")
fig.suptitle("Test performance, feature years 2023-2024 (134 companies, 54 distressed next year)", x=0.01, ha="left",
             fontsize=10, fontweight="bold")
fig.tight_layout()
P.save(fig, FIG / "fig10_model_comparison.png")
tt = t.set_index("model")
captions.append((10, "fig10_model_comparison.png", "Model comparison on the test years (F1, ROC-AUC and recall for the distressed class)",
    f"The random forest (chosen on validation) reached F1 {tt.loc['Random forest','f1']:.2f} and ROC-AUC {tt.loc['Random forest','roc_auc']:.2f}, far above the "
    f"current-ratio rule (F1 {tt.loc['Baseline: current ratio < 1','f1']:.2f}), but below the persistence baseline that uses this year's Z' "
    f"(F1 {tt.loc[PERSIST,'f1']:.2f}, ROC-AUC {tt.loc[PERSIST,'roc_auc']:.2f})."))
t[["model", "f1", "roc_auc", "recall"]]

,model,f1,roc_auc,recall
10,Baseline: current ratio < 1,0.137931,0.756250,0.074074
3,Decision tree,0.757282,0.852431,0.722222
1,Logistic regression,0.760000,0.884722,0.703704
7,XGBoost,0.764706,0.894213,0.722222
5,Random forest,0.778761,0.895139,0.814815
11,Baseline: persistence (Z' < 1.23 now),0.846154,0.953472,0.814815


## Figure 11: ROC curves

In [12]:
y = test_pred[TARGET]
curves = [("Random forest", "prob_Random forest"), ("XGBoost", "prob_XGBoost"), ("Logistic regression", "prob_Logistic regression"),
          ("Decision tree", "prob_Decision tree"), ("Baseline: persistence (Z' < 1.23 now)", "score_persistence"),
          ("Baseline: current ratio < 1", "score_current_ratio")]
fig, ax = plt.subplots(figsize=(5.6, 5.0))
ax.plot([0, 1], [0, 1], color=P.AXIS, lw=0.9, ls="-", zorder=1)
aucs = {}
for name, col in curves:
    fpr, tpr, _ = roc_curve(y, test_pred[col])
    aucs[name] = roc_auc_score(y, test_pred[col])
    st = P.MODEL_STYLE[name]
    ax.plot(fpr, tpr, color=st["color"], ls=st["ls"], lw=2.0 if name == "Random forest" else 1.5,
            label=f"{short.get(name, name)} (AUC {aucs[name]:.2f})", zorder=3)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
ax.set_aspect("equal")
ax.set_xlabel("False positive rate (healthy flagged as distressed)")
ax.set_ylabel("True positive rate (distressed caught)")
ax.set_title("ROC curves on the test years (2023-2024)")
ax.legend(loc="lower right")
P.save(fig, FIG / "fig11_roc_curves.png")
captions.append((11, "fig11_roc_curves.png", "ROC curves of all models and both baselines on the test years",
    f"Random forest, XGBoost and logistic regression lie close together (AUC {aucs['Logistic regression']:.2f} to {aucs['Random forest']:.2f}) and the "
    f"decision tree lower ({aucs['Decision tree']:.2f}); the persistence baseline "
    f"lies above all of them (AUC {aucs[PERSIST]:.2f}) and the current-ratio rule below (AUC {aucs['Baseline: current ratio < 1']:.2f})."))
aucs

{'Random forest': 0.8951388888888889,
 'XGBoost': 0.8942129629629629,
 'Logistic regression': 0.8847222222222222,
 'Decision tree': 0.8524305555555556,
 "Baseline: persistence (Z' < 1.23 now)": 0.9534722222222223,
 'Baseline: current ratio < 1': 0.75625}

## Figure 12: Feature importance (best model)

In [13]:
imp = importance.sort_values("perm_importance_mean")
names = imp["feature"].map(lambda f: nice.get(f, f.replace("_", " ").capitalize()))
fig, axes = plt.subplots(1, 2, figsize=(7.4, 4.6), sharey=True)
ax = axes[0]
ax.barh(names, imp["perm_importance_mean"], xerr=imp["perm_importance_std"], height=0.6, color="#2a78d6",
        error_kw={"ecolor": P.INK_2, "elinewidth": 0.8, "capsize": 2})
ax.axvline(0, color=P.AXIS, lw=0.8)
ax.set_title("Permutation importance")
ax.set_xlabel("Drop in test PR-AUC when shuffled")
ax.grid(axis="y", visible=False)
ax = axes[1]
ax.barh(names, imp["shap_share"] * 100, height=0.6, color="#9fc3ee")
ax.set_title("Share of mean |SHAP|")
ax.set_xlabel("Share of total (%)")
ax.set_xticks(np.arange(0, imp["shap_share"].max() * 100 + 2, 2))
ax.xaxis.set_major_formatter(pct)
ax.grid(axis="y", visible=False)
fig.suptitle("What drives the random forest's predictions (test years 2023-2024)", x=0.01, ha="left", fontsize=10, fontweight="bold")
fig.tight_layout()
P.save(fig, FIG / "fig12_feature_importance.png")
top3 = importance.head(3)["feature"].map(nice).tolist()
captions.append((12, "fig12_feature_importance.png", "Feature importance of the random forest (permutation importance and SHAP)",
    f"The most important ratios by permutation are {', '.join(top3)}; no single feature dominates (the largest SHAP share is "
    f"{importance['shap_share'].max():.0%}), so there is no sign that one variable leaks the label. The error bars (one standard deviation over 30 shuffles) are wide, so the order among the top six is uncertain."))
importance[["feature", "perm_importance_mean", "shap_share"]].head(8)

,feature,perm_importance_mean,shap_share
0,operating_margin,0.021946,0.133446
1,debt_to_equity,0.021775,0.082755
2,debt_ratio,0.019205,0.079401
3,log_total_assets,0.017776,0.079289
4,net_margin,0.015972,0.120002
5,roa,0.013975,0.111460
6,interest_coverage,0.008068,0.134582
7,gross_margin,0.003440,0.021891


## Figure 13: Predicted distress probability (walk-forward)

In [14]:
w = wf[wf["company"].notna()].copy()
thr = float(comparison.loc[(comparison["model"] == "Random forest") & (comparison["set"] == "test"), "threshold"].iloc[0])
wy = sorted(w["label_year"].unique())
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.axvspan(wy[-1] - 0.5, wy[-1] + 0.5, color="#f4f3ef", zorder=0, lw=0)
ax.annotate("Forecast\nfor 2026", xy=(wy[-1], 0.60), ha="center", va="center", fontsize=7.5, color=P.MUTED, fontstyle="italic")
ends = {}
for c in P.COMPANY_ORDER:
    d = w[w["company"] == c].set_index("label_year").reindex(wy)
    filled = (d[TARGET] == 1).values
    P.company_line(ax, d.index, d["prob_distressed_t1"].values, c, filled=filled)
    ends[c] = (d.index[-1], d["prob_distressed_t1"].iloc[-1])
P.ref_line(ax, thr, f"threshold {thr:.2f}")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Predicted probability of distress next year (random forest, out of sample)")
ax.set_ylabel("Probability of Z' < 1.23 in year t+1")
ax.set_xticks(wy); ax.set_xlim(wy[0] - 0.4, wy[-1] + 1.6)
ax.set_xlabel("Feature year t")
P.end_labels(ax, ends)
ax.annotate("Filled marker: actually distressed in t+1   Hollow: not distressed (or not yet known)",
            xy=(0, -0.2), xycoords="axes fraction", fontsize=7.5, color=P.INK_2)
P.save(fig, FIG / "fig13_case_probabilities.png")
pw = w.pivot_table(index="label_year", columns="company", values="prob_distressed_t1")
captions.append((13, "fig13_case_probabilities.png", "Walk-forward predicted probability of distress in the following year, 2018 to 2025",
    f"Each year is scored by a model trained only on earlier years. Nvidia stayed below {pw['Nvidia'].max():.0%}; Wolfspeed was above the threshold for every year "
    f"from the 2018 data ({pw.loc[2018,'Wolfspeed']:.0%}), including a false alarm for 2019, the year before its Z' first fell below 1.23 (2020); Intel's probability rose from {pw.loc[2022,'Intel']:.0%} (2022, missed) to "
    f"{pw.loc[2024,'Intel']:.0%} (2024); the 2025 forecasts for 2026 are {', '.join(f'{c} {pw.loc[2025,c]:.0%}' for c in P.COMPANY_ORDER)}."))
pw

company,AMD,Intel,Nvidia,Wolfspeed
label_year,,,,
2018,0.345158,0.139014,0.150079,0.781050
2019,0.321487,0.167697,0.095901,0.761301
2020,0.113914,0.183984,0.065162,0.728509
2021,0.071966,0.130911,0.123542,0.942284
2022,0.107141,0.277881,0.173264,0.831033
2023,0.142196,0.666713,0.114264,0.879364
2024,0.100459,0.850287,0.090551,0.844004
2025,0.055245,0.732808,0.091628,0.885069


## Captions

In [15]:
lines_md = ["# Figure captions", "",
            "Generated by `notebooks/06_figures_and_exports.ipynb`; every number in a takeaway comes from the project data.", ""]
for n, f, title, take in sorted(captions):
    lines_md += [f"**Figure {n}. {title}.** ({f})", "", take, ""]
(FIG / "captions.md").write_text("\n".join(lines_md), encoding="utf-8")
print(f"Saved {len(captions)} figures and captions to {FIG}")
for n, f, title, take in sorted(captions):
    print(f"\nFigure {n}: {title}\n  {take}")

Saved 13 figures and captions to G:\DJSCE\Fourth Year\FADS\Mini Proj\figures

Figure 1: Number of SIC 3674 companies per label year
  The sample shrinks from 93 companies in 2015 to 80 in 2025; 106 of 925 company-years are very small companies (revenue below USD 10 million), which are excluded from modelling.

Figure 2: Revenue, 2015 to 2025 (log scale)
  Nvidia's revenue grew from USD 5,010m to 215,938m and overtook Intel's in 2023; Intel fell from a peak of USD 79,024m (2021) to 52,853m, and Wolfspeed shrank from USD 1,633m to 758m. A log scale is used because Nvidia's 2025 revenue is about 285 times Wolfspeed's.

Figure 3: Net income, 2015 to 2025 (separate scales)
  Nvidia was profitable in every year; AMD made losses in 2015, 2016 and profits afterwards; Intel turned to losses in 2024 and 2025 (USD -18,756m in 2024); Wolfspeed lost money in every year from 2015 to 2025, reaching USD -1,609m in 2025.

Figure 4: Current ratio and quick ratio, with a reference line at 1
  All four co

# Phase 7: Dashboard exports (Power BI)

The dashboard is built in Power BI Desktop from CSVs written here (`src/exports.py`). Nothing is recalculated: every number is copied from the Phase 2-5 tables and only reshaped.

**Layout (star schema).** Two small *dimension* tables are the shared filters: `dim_company` (one row per CIK, with the latest filed name, so Cree appears as Wolfspeed) and `dim_year` (label years 2015-2025). The *fact* tables (ratios, P&L, Z' scores, predictions) each carry `cik`, `label_year` and a one-column key `cik_year`, because a Power BI relationship joins on a single column. A company selector and year slicer built on the dimensions then filter every chart.

**Conventions.** USD millions; ratios and margins as decimals (formatted as % in Power BI); flags as 0/1; blanks left empty so they are treated as missing, not as zero (a zero would pull medians down).

**Predictions.** The walk-forward random-forest probabilities use year-t ratios to predict distress in year t+1, so each row shows both `feature_year` and `target_year`. Risk bands: **Low** below 0.25, **Medium** 0.25 to the decision threshold, **High** at or above the threshold (0.494, chosen on validation in Phase 5), so High means exactly "predicted distressed". Low-revenue companies (revenue below USD 10m) were excluded from modelling, so they have no prediction.

This section runs on its own (it has its own imports).

In [1]:
import sys
import pandas as pd
sys.path.insert(0, "..")
from src import config
from src import exports as E

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

tables = E.build_all()
E.save_all(tables)
pd.DataFrame([{"file": f"{k}.csv", "rows": len(v), "columns": v.shape[1]} for k, v in tables.items()])

,file,rows,columns
0,dim_company.csv,146,10
1,dim_year.csv,11,1
2,dim_company_year.csv,925,5
3,fact_ratios.csv,925,34
4,fact_ratios_long.csv,12354,8
5,fact_pnl.csv,44,19
6,fact_zscore.csv,925,14
7,fact_predictions.csv,583,14
8,model_metrics.csv,12,18
9,model_metrics_long.csv,60,8


## Consistency checks

Keys must be unique and every fact row must find its company and year (otherwise Power BI silently drops it from filtered visuals). Row counts are compared with the data log, values with the Phase 3 ratio table, case-company values with the 10-K figures verified in Phase 3, and the High risk band with the model's own predictions.

In [2]:
checks = E.check_exports(tables)
print(f"{checks['ok'].sum()} of {len(checks)} checks passed")
checks

29 of 29 checks passed


,check,ok,detail
0,dim_company: one row per CIK,True,146 companies
1,dim_year: 2015-2025,True,11 years
2,dim_company_year: cik_year unique,True,925 rows
3,fact_ratios: cik_year unique,True,925 rows
4,fact_ratios: every cik_year in dim_company_year,True,0 orphan rows
5,fact_ratios: every CIK in dim_company,True,0 orphan rows
6,fact_ratios: every year in dim_year,True,years 2015-2025
7,fact_pnl: cik_year unique,True,44 rows
8,fact_pnl: every cik_year in dim_company_year,True,0 orphan rows
9,fact_pnl: every CIK in dim_company,True,0 orphan rows


## Reference values for the dashboard cards

`sector_by_year.csv` holds what the Sector overview cards should show for each year, for all companies and for operating companies only (the low-revenue toggle). Percent distressed is the same-year Z' label (Z' < 1.23) over company-years with a Z' score. After building the dashboard, compare its cards with this table.

In [3]:
s = tables["sector_by_year"]
show = s.assign(pct_distressed=(s["pct_distressed"] * 100).round(1),
                median_current_ratio=s["median_current_ratio"].round(2),
                median_net_margin=(s["median_net_margin"] * 100).round(1),
                median_roa=(s["median_roa"] * 100).round(1))
show.pivot(index="label_year", columns="scope",
           values=["companies", "pct_distressed", "median_current_ratio", "median_net_margin"])

companies                                          pct_distressed                                          median_current_ratio                                           \
scope      All companies Operating companies (revenue >= USD 10m)  All companies Operating companies (revenue >= USD 10m)        All companies Operating companies (revenue >= USD 10m)   
label_year                                                                                                                                                                                
2015                93.0                                     82.0           30.6                                     28.2                 3.40                                     3.57   
2016                83.0                                     77.0           49.4                                     46.6                 3.55                                     3.80   
2017                86.0                                     77.0           45.7                                     39.7                 3.40                                     3.65   
2018                85.0                                     75.0           40.2                                     34.2                 3.32                                     3.39   
2019                83.0                                     72.0           46.9                                     41.4                 2.81                                     2.82   
2020                81.0                                     71.0           39.7                                     35.3                 3.18                                     3.15   
2021                89.0                                     76.0           44.7                                     37.0                 3.13                                     3.04   
2022                83.0                                     73.0           37.8                                     30.6                 2.98                                     2.92   
2023                81.0                                     73.0           40.0                                     33.3                 3.21                                     3.21   
2024                81.0                                     71.0           51.2                                     44.3                 2.89                                     3.12   
2025                80.0                                     72.0           41.0                                     35.7                 2.65                                     2.69   

           median_net_margin                                           
scope          All companies Operating companies (revenue >= USD 10m)  
label_year                                                             
2015                     1.1                                      2.1  
2016                     1.7                                      2.0  
2017                     2.9                                      6.2  
2018                     0.7                                      3.3  
2019                     0.0                                      2.0  
2020                     4.5                                      6.5  
2021                     5.7                                      8.5  
2022                     5.6                                      7.7  
2023                    -1.9                                      1.2  
2024                   -11.7                                     -2.5  
2025                    -1.4                                      1.2

## Case companies: values to check on the drill-down and model pages

In [4]:
fr, fp, fz = tables["fact_ratios"], tables["fact_predictions"], tables["fact_zscore"]
case = tables["dim_company"].query("is_case_company == 1")[["cik", "company"]]
latest = (fr[fr["label_year"] == config.LABEL_YEAR_END][["cik", "revenue", "net_income", "current_ratio", "net_margin"]]
          .merge(fz[fz["label_year"] == config.LABEL_YEAR_END][["cik", "z_prime", "zone"]], on="cik")
          .merge(fp[fp["feature_year"] == config.LABEL_YEAR_END][["cik", "prob_distress_next_year", "risk_band"]], on="cik"))
case.merge(latest, on="cik").round(3)

,cik,company,revenue,net_income,current_ratio,net_margin,z_prime,zone,prob_distress_next_year,risk_band
0,1045810,Nvidia,215938.0,120067.0,3.905,0.556,5.261,safe,0.092,Low
1,2488,AMD,34639.0,4335.0,2.850,0.125,2.735,grey,0.055,Low
2,50863,Intel,52853.0,-267.0,2.017,-0.005,1.086,distress,0.733,High
3,895419,Wolfspeed,757.6,-1609.2,0.359,-2.124,-1.554,distress,0.885,High


In [5]:
# Risk bands by feature year (all scored companies) and the case companies' bands over time.
print(fp.pivot_table(index="feature_year", columns="risk_band", values="cik", aggfunc="count")[["Low", "Medium", "High"]])
fp.merge(case, on="cik").pivot(index="feature_year", columns="company", values="risk_band")[list(E.CASE_COLOURS)]

risk_band     Low  Medium  High
feature_year                   
2018           31      16    28
2019           29      13    30
2020           32      14    25
2021           36      12    28
2022           40      11    22
2023           34       7    32
2024           26      10    35
2025           27      14    31


company,Nvidia,AMD,Intel,Wolfspeed
feature_year,,,,
2018,Low,Medium,Low,High
2019,Low,Medium,Low,High
2020,Low,Low,Low,High
2021,Low,Low,Low,High
2022,Low,Low,Medium,High
2023,Low,Low,High,High
2024,Low,Low,High,High
2025,Low,Low,High,High


In [6]:
m = tables["model_metrics"]
m[m["dataset"] == "test"][["model", "precision", "recall", "f1", "roc_auc", "pr_auc", "threshold"]].round(3)

,model,precision,recall,f1,roc_auc,pr_auc,threshold
1,Logistic regression,0.826,0.704,0.760,0.885,0.844,0.770
3,Decision tree,0.796,0.722,0.757,0.852,0.756,0.933
5,Random forest,0.746,0.815,0.779,0.895,0.883,0.494
7,XGBoost,0.812,0.722,0.765,0.894,0.878,0.721
10,Baseline: current ratio < 1,1.000,0.074,0.138,0.756,0.715,NaN
11,Baseline: persistence (Z' < 1.23 now),0.880,0.815,0.846,0.953,0.939,NaN
